# NeuraRoads - 04 Calibration & Distance Demo

Show how camera calibration drives distance estimation and the BEV projection, and how tuning intrinsics / mounting geometry changes the numbers.

In [ ]:
import sys, os
from pathlib import Path
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
SRC = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd() / 'src'
sys.path.insert(0, str(SRC))
from utils.calibration import CameraCalibration
calib = CameraCalibration.from_config(frame_size=(1280, 720))
print('fx=%.1f fy=%.1f cx=%.1f cy=%.1f horizon_row=%d' % (calib.fx, calib.fy, calib.cx, calib.cy, calib.horizon_row()))

In [ ]:
# Distance vs bounding-box pixel height for each object class (known real heights).
import numpy as np, matplotlib.pyplot as plt
from utils.config_loader import load_config
heights = load_config('model_config')['distance_estimator']['real_heights_m']
names = load_config('model_config')['detector']['class_names']
px = np.linspace(20, 400, 100)
plt.figure(figsize=(9, 5))
for cid in [2, 9, 4, 3]:  # Car, Truck, Pedestrian, Bike
    d = [calib.distance_from_height(p, heights[cid]) for p in px]
    plt.plot(px, d, label=f'{names[cid]} ({heights[cid]}m)')
plt.xlabel('bbox height (px)'); plt.ylabel('distance (m)'); plt.legend()
plt.title('Pinhole distance vs box height @1280x720'); plt.grid(True); plt.show()

In [ ]:
# Project a synthetic set of objects into the BEV mini-map.
import numpy as np
from core.tracker import TrackedObject
from core.bev_transformer import BEVTransformer
cfg = load_config('model_config')
colors = {k: tuple(v) for k, v in cfg['visualization']['colors'].items()}
bev = BEVTransformer(cfg['bev'], calib, colors)
objs = []
for i, (x, dist) in enumerate([(500, 30), (700, 15), (900, 45)]):
    o = TrackedObject(i, np.array([x, 400, x+80, 520], np.float32), 2, 'Car', 0.9)
    o.distance_m = dist; o.color_key = 'getting_close'
    objs.append(o)
canvas = bev.render(objs)
import matplotlib.pyplot as plt, cv2
plt.figure(figsize=(4, 6)); plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.title('BEV'); plt.show()

In [ ]:
# To calibrate a real camera, run one of:
#   python src/scripts/calibrate_camera.py fov --hfov 70 --width 1920 --height 1080 --cam-height 1.25
#   python src/scripts/calibrate_camera.py chessboard --images calib/ --cols 9 --rows 6 --square 0.025
print('See src/scripts/calibrate_camera.py --help')